# CellCharter zone prediction

This notebook uses CellCharter (https://www.nature.com/articles/s41588-023-01588-4) to identify spatial neighborhoods in the Xenium dataset. For this analysis, we use the Flu acute and memory timepoint Xenium data in addition to the HDM data in order to facilitate identification of the TLS zone. This is because we observe, as expected, that TLS are more developed in the Flu lung tissue than the HDM lung tissue. 

**Pinned Environment:** [`conda_envs/cellcharter2_20250604.yml`](../conda_envs/cellcharter2_20250604.yml)

In [ ]:
import sys
import os
import math

In [ ]:
import anndata as ad
import squidpy as sq
import cellcharter as cc
import pandas as pd
import scanpy as sc
import scvi
import numpy as np

from lightning.pytorch import seed_everything

import matplotlib.pyplot as plt

import joblib

seed_everything(12345)
scvi.settings.seed = 12345

In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.titlesize'] = 20  

In [ ]:
# check gpu
import jax
print(jax.devices())  # Should list a GPU if successful

## Local file info

**Make sure to set** `DATA_DIR` in `config/paths.py` to point to the location of the downloaded Xenium outputs.

In [ ]:
from pathlib import Path
sys.path.append(str(Path.cwd().resolve().parents[0]))
from config.paths import DATA_DIR, BASE_OUTDIR

# data input folder - set DATA_DIR in config/paths.py to the location of the downloaded Xenium outputs
data_input_folder = DATA_DIR # spatial_tls_manuscript_data/Xenium_outputs
if not os.path.exists(data_input_folder):
    os.makedirs(data_input_folder)

# folder to save outputs
out_dir = os.path.join(BASE_OUTDIR, "cellcharter_analysis/outputs_mouselung_batchcorrect_xenseg_nb")
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [ ]:
# for adata
dir_list = [
    os.path.join(data_input_folder, 'output-XETG00195__0037002__TIS08779-002-003__20241016__230427'),
    os.path.join(data_input_folder, 'output-XETG00195__0036990__TIS08778-00-003__20241016__230427'),
    os.path.join(data_input_folder, 'output-XETG00195__0036990__TIS08780-002-003__20241016__230427'),
    os.path.join(data_input_folder, 'output-XETG00195__0037002__TIS08781-003-003__20241016__230427')
]

sample_list = ['TIS08779', 
               'TIS08778',
               'TIS08780',
               'TIS08781',
              ]

## Import adata for each sample

In [ ]:
def h5_to_adata(dir, sample_ID):
    """
    Load a spatial transcriptomics dataset from an h5 and corresponding cell metadata file into an AnnData object.

    Parameters
    ----------
    dir : str
        Path to the directory containing:
        - 'cell_feature_matrix.h5': the gene expression matrix in 10x HDF5 format.
        - 'cells.csv.gz': a gzipped CSV file with cell-level metadata, including spatial coordinates.
    
    sample_ID : str
        A string identifying the sample, added to the `.obs` dataframe of the AnnData object.

    Returns
    -------
    adata : anndata.AnnData
        The annotated single-cell data object with spatial coordinates and sample ID included.
    """
    
    # file names    
    h5_file = os.path.join(dir, 'cell_feature_matrix.h5')
    cells_file = os.path.join(dir, 'cells.csv.gz')
    
    # h5 to adata
    adata = sc.read_10x_h5(
        filename=h5_file
    )
    
    # cells file
    df = pd.read_csv(
        cells_file, 
        compression = 'gzip'
    )
    
    # set indx
    df.set_index(adata.obs_names, inplace=True)
    adata.obs = df.copy()
    
    # x, y
    adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].copy().to_numpy()

    # add sample
    adata.obs['sample'] = sample_ID
    
    return adata

In [ ]:
adata_list = []
for i, path in enumerate(dir_list):
    print('dir: ', path)
    sample_id = sample_list[i]
    print('sample id: ', sample_id)
    adata = h5_to_adata(path, sample_id)
    adata_list.append(adata)

### QC metrics 

In [ ]:
adata_list_qc = []
for i, adata_sample in enumerate(adata_list):
    sample_id = adata_sample.obs['sample'].iloc[0]
    print(i, sample_id)
    
     # filter out control probes and codewords
    filtered_genes = ~adata_sample.var_names.str.startswith(('Neg', 'Unassigned'))
    adata_sample = adata_sample[:, filtered_genes]

    # number of cells
    cell_num = adata_sample.obs.shape[0]

     # calculate QC
    sc.pp.calculate_qc_metrics(adata_sample, percent_top=(10, 20, 50, 150), inplace=True)

    # Visualize QC
    fig, axs = plt.subplots(1, 2, figsize=(15, 4))
    
    # Plot the first histogram using plt.hist
    axs[0].hist(adata_sample.obs["total_counts"], bins=math.ceil(cell_num/100), edgecolor='black')
    axs[0].set_title(sample_id + ", " + "total counts per cell")
    axs[0].set_xlim(0, 750)
    axs[0].set_xlabel('Total Counts')
    axs[0].set_ylabel('Frequency')
    
    # Plot the second histogram using plt.hist
    axs[1].hist(adata_sample.obs["n_genes_by_counts"], bins=math.ceil(cell_num/100), edgecolor='black')
    axs[1].set_title(sample_id + ", " + "Unique transcripts (genes) per cell")
    axs[1].set_xlim(0, 400)
    axs[1].set_xlabel('Unique Transcripts (genes)')
    axs[1].set_ylabel('Frequency')

    # # save figure
    fig.savefig(os.path.join(out_dir, sample_id + '_QC.png'))

    # append sample adata to list
    adata_list_qc.append(adata_sample)

### Concatenate all samples to single adata object and perform filtering, normalization, and log transformation

In [ ]:
# concatenate
adata = ad.concat(adata_list_qc, axis=0, merge='same', pairwise=True, index_unique='_')
adata.obs['sample'] = pd.Categorical(adata.obs['sample'])

# filter
sc.pp.filter_genes(adata, min_counts=10)
sc.pp.filter_cells(adata, min_counts=10)

# norm and log
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)

### add sample identities

In [ ]:
sample_labels = {
    'TIS08778': 'HDM_day3',
    'TIS08779': 'HDM_day30', 
    'TIS08780': 'PR8_day14', 
    'TIS08781': 'PR8_day40'
    }

# map to adata
adata.obs['sample_label'] = adata.obs['sample'].map(sample_labels)
adata.obs['sample_label'] = pd.Categorical(adata.obs['sample_label'], 
                                           categories = ['HDM_day3', 'HDM_day30', 'PR8_day14', 'PR8_day40'],
                                           ordered=True)
                                    

## scVI dimensionality reduction and cellcharter


In [ ]:
def run_cellcharter(adata, out_dir, n_layers=3, low_zone=8, high_zone=13):
    """
    Run CellCharter spatial clustering on a spatial AnnData object using scVI embeddings.

    Parameters
    ----------
    adata : anndata.AnnData
        An AnnData object with:
        - Spatial coordinates in `adata.obsm['spatial']`
        - scVI embeddings in `adata.obsm['X_scVI']`
        - A column named `'sample'` in `adata.obs` (used as the library key).

    out_dir : str
        Directory where the results (clustered AnnData and AutoK model) will be saved.

    n_layers : int, optional (default=3)
        Number of spatial neighborhood layers to aggregate in CellCharter's preprocessing.

    low_zone : int, optional (default=8)
        Lower bound of the number of clusters to test with AutoK.

    high_zone : int, optional (default=13)
        Upper bound of the number of clusters to test with AutoK.

    Returns
    -------
    adata : anndata.AnnData
        The updated AnnData object with new cluster assignments added to `adata.obs`
        as columns named `cluster_cellcharter_<k>` for each k in [low_zone, high_zone].

    Outputs
    -------
    Saves the following files to `out_dir`:
    - `adata_scvi_cellcharter_nlayer<n_layers>_k<low_zone>-<high_zone>.h5ad`
    - `autok_cellcharter_nlayer<n_layers>_k<low_zone>-<high_zone>.pkl`
    """
    
    # Neighbor graph and zone ID
    print('Starting neighbors and CellCharter zone identification...')
    sq.gr.spatial_neighbors(adata, library_key='sample', coord_type='generic', delaunay=True, spatial_key='spatial')
    cc.gr.remove_long_links(adata)
    cc.gr.aggregate_neighbors(adata, n_layers=n_layers, use_rep='X_scVI', out_key='X_cellcharter', sample_key='sample')
    
    # Run CellCharter AutoK
    print(f'Running AutoK clustering for k={low_zone} to k={high_zone}...')
    model_params = {
        'random_state': 42,
        'trainer_params': {
            'accelerator': 'gpu',
            'enable_progress_bar': True
        }
    }
    autok = cc.tl.ClusterAutoK(
        n_clusters=(low_zone, high_zone),
        max_runs=10,
        model_params=model_params
    )
    
    autok.fit(adata, use_rep='X_cellcharter')
    
    for k in range(low_zone, high_zone + 1):
        colname = f'cluster_cellcharter_{k}'
        adata.obs[colname] = autok.predict(adata, use_rep='X_cellcharter', k=k)
        print(f' → {colname} added')

    # Build filename suffix
    suffix = f'nlayer{n_layers}_k{low_zone}-{high_zone}'

    # Save outputs with dynamic filenames
    adata_path = os.path.join(out_dir, f'adata_scvi_cellcharter_{suffix}.h5ad')
    model_path = os.path.join(out_dir, f'autok_cellcharter_{suffix}.pkl')

    adata.write_h5ad(adata_path, compression='gzip')
    joblib.dump(autok, model_path)

    print(f'\nSaved: {adata_path}')
    print(f'Saved: {model_path}')

    return adata


### Run scvi followed by cellcharter in same block

In [ ]:
# scvi model

scvi.settings.seed = 12345
scvi.model.SCVI.setup_anndata(
    adata, 
    layer="counts", 
    batch_key='sample'
)

model = scvi.model.SCVI(adata, n_layers=2, n_latent=30, gene_likelihood="nb")

print('starting model training')
model.train(early_stopping=True, accelerator="gpu",
           max_epochs = 20 # April 2025 adding max epochs = 20; without setting this, it runs only 7
           )

# save model
model.save(os.path.join(out_dir, 'models') , prefix='mouselung_v1_', overwrite=True)

# add scVI latent dimensions to adata.obsm
adata.obsm['X_scVI'] = model.get_latent_representation(adata).astype(np.float32)
adata.obsm['X_scVI'].shape

# save adata with scvi dims
filename = os.path.join(out_dir, 'adata_scvi.h5ad')
adata.write_h5ad(filename, compression='gzip')

# neighbors
print('starting scvi neighbors')
sc.pp.neighbors(adata, use_rep="X_scVI", key_added='neighbors_scvi')
# save adata with scvi dims
filename = os.path.join(out_dir, 'adata_scvi.h5ad')
adata.write_h5ad(filename, compression='gzip')

# umap on scvi dims as alternative to mde
print('starting umap')
sc.tl.umap(adata, neighbors_key='neighbors_scvi')
# save umap dims to csv
umap_df = pd.DataFrame(adata.obsm['X_umap'], index=adata.obs_names, columns=['umap1', 'umap2'])
filename = os.path.join(out_dir, 'adata_scvi_umap_coords.csv')
umap_df.to_csv(filename)

# leiden
print('starting leiden res1')
sc.tl.leiden(adata, resolution = 1, key_added='leiden_scvi_res1', neighbors_key = 'neighbors_scvi', n_iterations=2)
# save leiden to csv
filename = os.path.join(out_dir, 'adata_scvi_leiden_scvi_res1.csv')
adata.obs[['leiden_scvi_res1']].to_csv(filename)


## Cellcharter
adata = run_cellcharter(adata, out_dir, low_zone=8, high_zone=13)

## Visualize cellcharter results

In [ ]:
adata = sc.read_h5ad(os.path.join(out_dir, 'adata_scvi_cellcharter_nlayer3_k8-13.h5ad'))

In [ ]:
# load trained scvi model
model = scvi.model.SCVI.load(os.path.join(out_dir, 'models'), adata=adata, prefix = 'mouselung_v1_')

In [ ]:
history = model.history["elbo_train"]
plt.plot(range(1, len(history) + 1), history, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs. Epochs")
plt.show()

In [ ]:
sc.pl.embedding(adata, basis='X_umap', color=['sample_label'], frameon=False)

In [ ]:
major_markers_short = [  
    "Cd79a", # B cell
    "Cd3e", "Cd8a", "Cd4", # T cell
    "Cd34",  # BEC
    "Prox1", "Lyve1", # LEC
    "Epcam", # epithelial
    "Cxcr5",
    "Foxp3",
    "Nkg7"
]

major_markers_extra_short = [  
    "Cd79a", # B cell
    "Cd3e", # T cell
    "Lyve1", # LEC
    "Epcam", # epithelial
]



In [ ]:
sc.pl.embedding(adata, basis='X_umap', color=major_markers_short, frameon=False)

## Cellcharter visualization

In [ ]:
import random
import colorcet
import seaborn as sns
unique_values = sorted(adata.obs["cluster_cellcharter_13"].unique().tolist())
palette = sns.color_palette(colorcet.glasbey, n_colors=len(unique_values)) 
random.shuffle(palette)

zone_palette_mapped = {val: color for val, color in zip(unique_values, palette)}

In [ ]:
for sample_label in ['HDM_day3', 'HDM_day30']:
    adata_plot = adata[adata.obs['sample_label']==sample_label, :]
    sc.pl.embedding(adata_plot, basis='spatial', color=['cluster_cellcharter_8', 'cluster_cellcharter_9', 'cluster_cellcharter_10', 'cluster_cellcharter_11', 'cluster_cellcharter_12', 'cluster_cellcharter_13'],
                                                        palette=zone_palette_mapped, frameon=False, size=1, ncols=4)

## Save cellcharter zone labels to csv

In [ ]:
# save xenium labels to csv
filename = os.path.join(out_dir, 'adata_scvi_cellcharter_zone_cols.csv')
adata.obs[['cluster_cellcharter_8', 'cluster_cellcharter_9', 'cluster_cellcharter_10', 
            'cluster_cellcharter_11', 'cluster_cellcharter_12', 'cluster_cellcharter_13']].to_csv(filename)

In [ ]:
import session_info
print('active conda environment: ', os.path.basename(sys.prefix))
session_info.show()